# NLP Lecture Notebook: From Transformers to Modern LLMs

Welcome to the practical companion for our lecture on modern NLP. In this notebook, we will implement the core concepts discussed, from tokenization to interacting with local and cloud-based Large Language Models.

## 0. Setup

First, we need to install the necessary Python libraries. We will use:
- `transformers` and `torch` for low-level NLP tasks like tokenization.
- `openai` to interact with the OpenAI API.
- `ollama` to interact with models running locally.

After running the cell below, you may need to restart your runtime environment.
- `pydantic` for **structured outputs** (section 3.4)
- `mcp[cli]` for the **Model Context Protocol** section (section 5)


In [14]:
# Use a ! to run shell commands from inside the notebook
!pip install -q transformers torch openai pydantic "mcp[cli]" ollama python-dotenv

print("Libraries installed successfully!")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.115.14 requires starlette<0.47.0,>=0.40.0, but you have starlette 1.3.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Libraries installed successfully!


In [2]:
# Load openai key from .env file
from dotenv import load_dotenv

load_dotenv()

False

## 1. Tokenization in Practice

This section corresponds to **Part 3** of the presentation.

As we discussed, models don't see words; they see tokens. Let's make this concept concrete. We will use a tokenizer from the Hugging Face `transformers` library to see how a sentence is broken down into subwords and then converted into numerical IDs. We'll use the classic `gpt2` tokenizer for this example.

In [3]:
from transformers import AutoTokenizer

# Load a pre-trained tokenizer. GPT-2 is a great example.
tokenizer = AutoTokenizer.from_pretrained("gpt2")

sentence = "The Transformer architecture is unbelievably powerful."

# 1. Tokenize the sentence into subword strings
tokens = tokenizer.tokenize(sentence)

print(f"Original Sentence: {sentence}")
print("-" * 30)
print(f"Subword Tokens: {tokens}")
print("-" * 30)


# 2. Encode the sentence into integer IDs
# This is what the model actually receives as input
input_ids = tokenizer.encode(sentence)

print(f"Input IDs: {input_ids}")
print("-" * 30)

# You can also decode the IDs back to a string to verify
decoded_sentence = tokenizer.decode(input_ids)
print(f"Decoded Sentence: {decoded_sentence}")

/home/x/.pyenv/versions/torch_compatible/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original Sentence: The Transformer architecture is unbelievably powerful.
------------------------------
Subword Tokens: ['The', 'ĠTrans', 'former', 'Ġarchitecture', 'Ġis', 'Ġunbelievably', 'Ġpowerful', '.']
------------------------------
Input IDs: [464, 3602, 16354, 10959, 318, 48943, 3665, 13]
------------------------------
Decoded Sentence: The Transformer architecture is unbelievably powerful.


## 2. Hands-on Prompting Techniques

This section corresponds to **Part 5** of the presentation.

Now that we have our API client ready, we can explore how to "program" the model using different prompting styles. We'll be using a modern chat model (`gpt-4o`), which has been fine-tuned to be excellent at following instructions.

To make our code cleaner, let's first create a small helper function to handle the API calls.

In [3]:
import openai

# Initialize the OpenAI client
# Make sure you have set your OpenAI API key in the environment variable OPENAI_API_KEY
client = openai.OpenAI()


def get_response(prompt, model="gpt-4o"):
    """
    A helper function to get a response from the OpenAI API.
    """     
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                # We can add a system message to set the model's behavior
                {"role": "system", "content": "You are a helpful assistant."},
                # The user's prompt
                {"role": "user", "content": prompt}
            ],
            temperature=0.7, # A value between 0 and 2. Higher values make the output more random.
            max_tokens=256   # The maximum number of tokens to generate.
        )
        return response.choices[0].message.content
    except Exception as e:
        # Handle potential API errors, e.g., invalid key
        return f"An error occurred: {e}"

# Let's test the helper function
test_prompt = "Hello, world!"
test_response = get_response(test_prompt)
print(f"Test Response: {test_response}")

Test Response: Hello! How can I assist you today?


### 2.1 Zero-Shot Prompting

This is the most basic form of prompting. We simply give the model a direct instruction without any prior examples.

Let's try a simple sentiment classification task.

In [4]:
zero_shot_prompt = """
Classify the following movie review as either 'Positive' or 'Negative'.

Review: 'This movie was a complete masterpiece! The acting was incredible and the story was unforgettable.'
"""

response = get_response(zero_shot_prompt)

print("--- Zero-Shot Classification ---")
print(f"Prompt:\n{zero_shot_prompt}")
print(f"\nModel Response:\n{response}")

--- Zero-Shot Classification ---
Prompt:

Classify the following movie review as either 'Positive' or 'Negative'.

Review: 'This movie was a complete masterpiece! The acting was incredible and the story was unforgettable.'


Model Response:
Positive


### 2.2 Few-Shot Prompting

For more complex tasks or when we need a very specific output format, we can provide a few examples ("shots") within the prompt itself. This is called in-context learning.

Here, we'll ask the model to extract specific data from a sentence and format it as `Attribute: Value`.

In [5]:
few_shot_prompt = """
Extract the Technology, Company, and Year from the following sentences.

Sentence: "In 1991, Guido van Rossum created Python."
Technology: Python, Company: N/A, Year: 1991
###
Sentence: "The first iPhone was released by Apple in 2007."
Technology: iPhone, Company: Apple, Year: 2007
###
Sentence: "In 2023, OpenAI released GPT-4."
Technology:
"""

response = get_response(few_shot_prompt)

print("--- Few-Shot Extraction ---")
print(f"Prompt:\n{few_shot_prompt}")
print(f"\nModel Response:\n{response}")

--- Few-Shot Extraction ---
Prompt:

Extract the Technology, Company, and Year from the following sentences.

Sentence: "In 1991, Guido van Rossum created Python."
Technology: Python, Company: N/A, Year: 1991
###
Sentence: "The first iPhone was released by Apple in 2007."
Technology: iPhone, Company: Apple, Year: 2007
###
Sentence: "In 2023, OpenAI released GPT-4."
Technology:


Model Response:
GPT-4, Company: OpenAI, Year: 2023


### 2.3 Chain-of-Thought (CoT) Prompting

When a problem requires multiple steps to solve, we can encourage the model to "think step by step." This often leads to more accurate results as it forces the model to work through its reasoning process before giving a final answer.

Let's try a simple word problem.

In [6]:
cot_prompt = """
Q: A grocery store has 5 boxes of apples, with 12 apples in each box. They sell 20 apples. Then, they receive a new shipment of 3 boxes, each containing 10 apples. How many apples do they have now?

Let's think step by step to find the answer.
"""

response = get_response(cot_prompt)

print("--- Chain-of-Thought Reasoning ---")
print(f"Prompt:\n{cot_prompt}")
print(f"\nModel Response:\n{response}")

--- Chain-of-Thought Reasoning ---
Prompt:

Q: A grocery store has 5 boxes of apples, with 12 apples in each box. They sell 20 apples. Then, they receive a new shipment of 3 boxes, each containing 10 apples. How many apples do they have now?

Let's think step by step to find the answer.


Model Response:
To determine how many apples the grocery store has now, let's break it down step by step:

1. **Initial Number of Apples:**
   - The store starts with 5 boxes of apples, with each box containing 12 apples.
   - Therefore, the initial number of apples is \(5 \times 12 = 60\) apples.

2. **Apples Sold:**
   - The store sells 20 apples.
   - After selling, the number of apples remaining is \(60 - 20 = 40\) apples.

3. **New Shipment:**
   - They receive a new shipment of 3 boxes, each containing 10 apples.
   - Therefore, the number of apples in the new shipment is \(3 \times 10 = 30\) apples.

4. **Total Number of Apples Now:**
   - Add the apples from the new shipment to the apples rema

## 3. Reliable Outputs: Structured Data and Tool Use

This section corresponds to **Part 6** of the presentation.

While natural language is great for chat, applications need reliable, machine-readable data. Here we'll explore two powerful techniques for getting structured output.

### 3.1 JSON Mode

Modern APIs can guarantee that the model's output will be a syntactically correct JSON object. This is incredibly useful for preventing errors and simplifying your code.

In [7]:
import json

# Check if the client is initialized
if 'client' not in globals():
    print("OpenAI client not initialized. Please run the API key setup cell.")
else:
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            # This is the key parameter to enable JSON mode
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You are a helpful assistant designed to output JSON."},
                {"role": "user", "content": "Extract the user's name, age, and city from this sentence: 'Amanda, who is 34, just moved to sunny San Diego.'"}
            ]
        )

        output_str = response.choices[0].message.content
        print("--- Raw Model Output (a guaranteed JSON string) ---")
        print(output_str)

        # Because the output is guaranteed to be valid JSON, we can parse it directly
        parsed_json = json.loads(output_str)

        print("\n--- Parsed JSON Object ---")
        print(parsed_json)
        print(f"\nUser's Name: {parsed_json.get('name')}")

    except Exception as e:
        print(f"An error occurred: {e}")

--- Raw Model Output (a guaranteed JSON string) ---
{
  "name": "Amanda",
  "age": 34,
  "city": "San Diego"
}

--- Parsed JSON Object ---
{'name': 'Amanda', 'age': 34, 'city': 'San Diego'}

User's Name: Amanda


### 3.2 Tool Use / Function Calling

This is the full implementation of the ReAct principle. We give the LLM a set of "tools" (our Python functions) that it can ask our application to run. This allows the model to access real-time data or perform actions.

Let's walk through the full loop: `Define -> Request -> Execute -> Synthesize`.

In [8]:
import json

# Check if the client is initialized
if 'client' not in globals():
    print("OpenAI client not initialized. Please run the API key setup cell.")
else:
    # Step 1: Define a local Python function that the model can "call"
    def get_mock_stock_price(ticker_symbol: str):
        """Gets the mock stock price for a given ticker symbol."""
        print(f"--- Application: Running get_mock_stock_price for {ticker_symbol} ---")
        if "NVDA" in ticker_symbol.upper():
            return json.dumps({"ticker": "NVDA", "price": "125.50 USD"})
        elif "GOOG" in ticker_symbol.upper():
            return json.dumps({"ticker": "GOOG", "price": "178.20 USD"})
        else:
            return json.dumps({"ticker": ticker_symbol, "price": "unknown"})

    # Step 2: Define the "tool" specification for the API call
    tools = [
        {
            "type": "function",
            "function": {
                "name": "get_mock_stock_price",
                "description": "Get the current stock price for a specific ticker symbol",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "ticker_symbol": {
                            "type": "string",
                            "description": "The stock ticker symbol, e.g., 'NVDA'",
                        },
                    },
                    "required": ["ticker_symbol"],
                },
            },
        }
    ]
    
    # We will use this list to keep track of the conversation
    messages = [{"role": "user", "content": "What is the current stock price of NVIDIA (NVDA)?"}]

    # Step 3: Make the first call to the model
    try:
        print("--- Step 3: Sending user prompt to the model ---")
        first_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        
        response_message = first_response.choices[0].message
        
        # Step 4: Check if the model wants to call a tool
        if response_message.tool_calls:
            print("--- Step 4: Model decided to call a tool ---")
            
            # Append the assistant's response to the message history
            messages.append(response_message)
            
            # Execute the function(s)
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                if function_name == "get_mock_stock_price":
                    function_args = json.loads(tool_call.function.arguments)
                    function_response = get_mock_stock_price(
                        ticker_symbol=function_args.get("ticker_symbol")
                    )
                    
                    # Append the function's response to the message history
                    messages.append(
                        {
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "name": function_name,
                            "content": function_response,
                        }
                    )
            
            # Step 5: Make the second call to the model with the tool's response
            print("--- Step 5: Sending tool response back to the model for synthesis ---")
            second_response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
            )
            
            print("\n--- Final Answer ---")
            print(second_response.choices[0].message.content)

        else:
            print("--- Model responded directly ---")
            print(response_message.content)

    except Exception as e:
        print(f"An error occurred: {e}")

--- Step 3: Sending user prompt to the model ---
--- Step 4: Model decided to call a tool ---
--- Application: Running get_mock_stock_price for NVDA ---
--- Step 5: Sending tool response back to the model for synthesis ---

--- Final Answer ---
The current stock price of NVIDIA (NVDA) is 125.50 USD.


### 3.3 From a single call to an agent loop

The cell above made the model call **one** tool, **once**. A real *agent* repeats the cycle — call the model, run whatever tools it asks for, feed the results back, and loop — until the model stops asking for tools and just answers. Give it more than one tool and it will **chain** them by itself.

In [9]:
import json

# In the cell above, the model called ONE tool, ONCE. Real agents loop:
#   call model -> if it asks for tools, run them -> feed results back -> repeat
#   until the model answers with plain text (no more tool calls).
# Below we give it TWO tools so it has to CHAIN them.

def get_stock_price(ticker: str):
    prices = {"NVDA": 125.50, "GOOG": 178.20, "AAPL": 229.00}
    return {"ticker": ticker.upper(), "price_usd": prices.get(ticker.upper(), 0.0)}

def calculator(expression: str):
    # NEVER use eval() on real user input. Fine for a controlled demo.
    return {"expression": expression, "result": eval(expression)}

# One tool spec per function, in OpenAI's function-calling format
TOOLS = [
    {"type": "function", "function": {
        "name": "get_stock_price",
        "description": "Get the latest share price (USD) for a ticker symbol",
        "parameters": {"type": "object",
            "properties": {"ticker": {"type": "string", "description": "e.g. NVDA"}},
            "required": ["ticker"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression like '125.5 * 10'",
        "parameters": {"type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"]}}},
]

# A dispatcher maps the tool NAME the model picks to the real Python function
DISPATCH = {"get_stock_price": get_stock_price, "calculator": calculator}

def run_agent(user_prompt, model="gpt-4o", max_steps=5):
    messages = [{"role": "user", "content": user_prompt}]
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, tool_choice="auto")
        msg = resp.choices[0].message
        messages.append(msg)                      # keep the assistant turn

        if not msg.tool_calls:                    # no tool wanted -> we're done
            return msg.content

        for tc in msg.tool_calls:                 # run every tool the model asked for
            args = json.loads(tc.function.arguments)
            print(f"  [step {step}] -> {tc.function.name}({args})")
            result = DISPATCH[tc.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result)})
    return "Stopped: hit max_steps."

print(run_agent("How much is 10 shares of NVDA worth right now?"))

  [step 0] -> get_stock_price({'ticker': 'NVDA'})
  [step 1] -> calculator({'expression': '125.5 * 10'})
10 shares of NVDA are worth $1255.00 right now.


**▶️ Output (pre-run)**

```
  [step 0] -> get_stock_price({'ticker': 'NVDA'})
  [step 1] -> calculator({'expression': '125.5 * 10'})
10 shares of NVIDIA (NVDA) at $125.50 each are worth $1,255.00.
```

Notice the model **chained two tools on its own**: first it fetched the price, then — seeing it needed arithmetic — it called the calculator, and only *then* wrote the final answer. That loop (call → run tools → feed back → repeat) is the core of every "agent". Frameworks like LangGraph just wrap this loop with state, memory, and error handling.

### 3.4 Guaranteed structure: Structured Outputs

JSON mode (section 3.1) promises *valid JSON* but not *which fields*. **Structured Outputs** go further: you hand the model a schema (here, a Pydantic class) and the API **guarantees** the reply matches it — right field names, right types — so you can load it straight into your app with no defensive parsing.

In [10]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    title: str
    date: str          # ISO date, e.g. 2025-07-26
    attendees: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-4o-2024-08-06",                # structured outputs need a recent model
    messages=[
        {"role": "system", "content": "Extract the event details."},
        {"role": "user", "content":
            "Sync with Ada and Lin about the launch on July 26, 2025."},
    ],
    response_format=CalendarEvent,            # <-- pass the schema, not a string
)

event = completion.choices[0].message.parsed  # already a typed CalendarEvent object
print(type(event))
print(event.title, "|", event.date, "|", event.attendees)

<class '__main__.CalendarEvent'>
Sync with Ada and Lin | 2025-07-26 | ['Ada', 'Lin']


**▶️ Output (pre-run)**

```
<class '__main__.CalendarEvent'>
Launch sync | 2025-07-26 | ['Ada', 'Lin']
```

`event` is a real Python object, not a string we had to `json.loads()` and hope for the best. This is the reliable way to turn free text into data your code can trust.

## 4. The LLM Ecosystem: API vs. Local

This final section corresponds to **Part 7** of the presentation. We'll look at the two primary ways to interact with models, which we've already set up: via a cloud API and by running a model locally on our own machine.

### 4.1 API Access (Recap)

All of the previous examples that used the `openai` client are examples of API access. We sent a request to a powerful model hosted on OpenAI's servers and received a response. This is the most common way to access state-of-the-art models without needing powerful hardware.

---

### 4.2 Local Models with Ollama

The second approach is to run an open-source model directly on your computer. This gives you complete privacy and control. We'll use **Ollama**, a fantastic tool that makes this process simple.

**Setup Instructions:**

1.  **Install Ollama:** If you haven't already, go to [ollama.com](https://ollama.com) and download the application for your operating system.
2.  **Run Ollama:** Make sure the Ollama application is running in the background.
3.  **Pull a Model:** Open your terminal (not in this notebook) and run the following command to download a small, efficient model. We'll use `gemma:2b`, a 2-billion parameter model from Google that is great for demonstrations.

```bash
ollama pull gemma:2b

In [5]:
import ollama

# This code assumes the Ollama application is running on your machine.
# If it is not, this cell will raise an error.

try:
    response = ollama.chat(
        # This model name must match one you have pulled with `ollama pull`
        model='gemma:2b',
        messages=[
            {'role': 'user', 'content': 'In one short sentence, why is learning about Transformers important?'},
        ],
    )
    
    print("--- Response from local 'gemma:2b' model ---")
    print(response['message']['content'])

except Exception as e:
    print("An error occurred. Is the Ollama application running on your computer?")
    print(f"Error details: {e}")

--- Response from local 'gemma:2b' model ---
Sure, here's a short sentence explaining why learning about Transformers is important:

Transformers are a revolutionary field in artificial intelligence that has the potential to revolutionize how we process and understand language, leading to significant advancements in natural language processing (NLP), machine translation, and other related fields.


## 5. Model Context Protocol (MCP)

Our agent above can call **our** Python functions. But every team re-writes the same connectors — one for Google Drive, one for GitHub, one for Postgres — and once per framework. With **M** apps and **N** tools that is **M × N** integrations.

**MCP** (Model Context Protocol, an open standard introduced by Anthropic in late 2024 and now supported across the ecosystem — IDEs, Claude Desktop, the OpenAI Agents SDK, and more) turns this into **M + N**: a tool author writes **one** MCP *server*; any MCP-aware app can use it.

**Architecture**
- **Host** — the LLM app (your agent, an IDE, a chat client).
- **Client** — lives inside the host; keeps a 1:1 connection to one server.
- **Server** — exposes capabilities. Three kinds:
  - **Tools** — functions the *model* can call (like our `get_stock_price`).
  - **Resources** — read-only data/context the *app* pulls in (files, rows, configs).
  - **Prompts** — reusable prompt templates the *user* can trigger.
- **Transports** — `stdio` (server runs as a local subprocess) or **streamable HTTP** (remote servers).

The payoff: the model doesn't care whether a tool is local Python or a remote service — it just sees a list of tools with schemas. That's exactly the shape our agent loop already speaks.

### 5.1 A minimal MCP server

Install the SDK with `pip install "mcp[cli]"` (it's in the Setup cell). The cell below starts with the `%%writefile` magic, so **running it saves the code to `weather_server.py`** on disk — it does **not** start the server inside the notebook (that would clash with Jupyter's event loop). The client in 5.2 launches that file as a subprocess. `FastMCP` turns each decorated function into a fully-described tool automatically, reading the type hints and docstring to build the schema.

In [16]:
%%writefile weather_server.py
# weather_server.py  --  run as: python weather_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather")

@mcp.tool()
def get_forecast(city: str) -> str:
    """Return a short weather forecast for a city."""
    data = {"Kyiv": "18C, partly cloudy",
            "London": "12C, rain",
            "Tokyo": "24C, clear"}
    return data.get(city, f"No forecast available for {city}")

@mcp.resource("config://cities")
def supported_cities() -> str:
    """The cities this server has data for."""
    return "Kyiv, London, Tokyo"

if __name__ == "__main__":
    mcp.run(transport="stdio")   # talk over stdin/stdout as a subprocess

Overwriting weather_server.py


### 5.2 An MCP client: discover and call tools

The client launches the server as a subprocess, performs the MCP handshake (`initialize`), then `list_tools()` / `call_tool()`. Note we never imported `get_forecast` — we discovered it over the protocol.

> In a notebook there is already a running event loop, so use `await main()` in a cell. As a script, wrap it in `asyncio.run(main())`.

In [17]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command="python", args=["weather_server.py"])

async def main():
    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("Discovered tools:", [t.name for t in tools.tools])

            result = await session.call_tool("get_forecast", {"city": "Kyiv"})
            print("get_forecast('Kyiv') ->", result.content[0].text)

await main()      # in a .py script use: asyncio.run(main())

Discovered tools: ['get_forecast']
get_forecast('Kyiv') -> 18C, partly cloudy


**▶️ Output (pre-run)**

```
Discovered tools: ['get_forecast']
get_forecast('Kyiv') -> 18C, partly cloudy
```

### 5.3 Wiring MCP tools into our OpenAI agent

An MCP tool already carries a JSON-Schema (`inputSchema`) — the same shape OpenAI's function-calling expects. So bridging them is one small adapter, and the agent loop from 3.3 is reused unchanged: when the model asks for a tool, we forward the call to the MCP server instead of a local dict.

In [18]:
import json

def mcp_to_openai_tools(mcp_tools):
    """Translate MCP tool definitions into OpenAI function-tool specs."""
    return [{"type": "function", "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema}}
            for t in mcp_tools.tools]

async def run_mcp_agent(user_prompt, model="gpt-4o", max_steps=5):
    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = mcp_to_openai_tools(await session.list_tools())

            messages = [{"role": "user", "content": user_prompt}]
            for _ in range(max_steps):
                resp = client.chat.completions.create(
                    model=model, messages=messages, tools=tools, tool_choice="auto")
                msg = resp.choices[0].message
                messages.append(msg)
                if not msg.tool_calls:
                    return msg.content
                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments)
                    # forward the call OVER MCP instead of to a local function:
                    out = await session.call_tool(tc.function.name, args)
                    messages.append({"role": "tool", "tool_call_id": tc.id,
                                     "content": out.content[0].text})
            return "Stopped: hit max_steps."

print(await run_mcp_agent("What's the weather in Tokyo and should I bring a coat?"))

[06/29/26 18:12:19] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=3099411;file:///home/pc/.pyenv/versions/ds_camp/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=3099412;file:///home/pc/.pyenv/versions/ds_camp/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[06/29/26 18:12:20] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=3099417;file:///home/pc/.pyenv/versions/ds_camp/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=3099418;file:///home/pc/.pyenv/versions/ds_camp/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

The weather in Tokyo is currently clear with a temperature of 24°C. It's quite warm, so you likely won't need a coat.


**▶️ Output (pre-run)**

```
Tokyo is 24C and clear right now, so you won't need a coat - light clothing is fine.
```

The model called `get_forecast` **through MCP**, got `24C, clear`, and reasoned about the coat. To give it real-world powers — files, GitHub, Slack, a database — you now just point the client at an existing MCP server. No new agent code. That M + N leverage is why MCP caught on so fast.

# Homework

Everything below runs **locally against Ollama** — no OpenAI API key required. Ollama exposes an OpenAI-compatible endpoint, so we reuse the exact same SDK and the agent loop from section 3.3.

**Before you start** (once, from a terminal):
```
ollama serve              # if it isn't already running
ollama pull llama3.1      # a TOOL-CAPABLE model (qwen2.5 / llama3.2 also work; gemma:2b does NOT)
```
Run the setup cell, then fill in the `# TODO`s in each task.

In [8]:
# --- Homework setup: local model via Ollama, no OpenAI key needed ---
import json
from openai import OpenAI

# Ollama speaks the OpenAI API on localhost:11434/v1 -> reuse the same client.
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
OLLAMA_MODEL = "llama3.1"          # must support tool calling

# Smoke test (should print something like "ready"):
r = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": "Reply with one word: ready"}])
print("Model says:", r.choices[0].message.content)

# The agent loop from section 3.3, parameterised for any client/model:
def run_agent_local(user_prompt, tools, dispatch,
                    client=ollama_client, model=OLLAMA_MODEL, max_steps=5):
    messages = [{"role": "user", "content": user_prompt}]
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=tools, tool_choice="auto")
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"  [step {step}] -> {tc.function.name}({args})")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(dispatch[tc.function.name](**args))})
    return "Stopped: hit max_steps."

Model says: Ready!


**▶️ Expected output** (this cell runs as-is)

```
Model says: ready
```

If you get a connection error, Ollama isn't running (`ollama serve`); if the model isn't found, run `ollama pull llama3.1` first.

### Task 1 — Add a third tool and watch the model chain

Give the agent a **currency converter** on top of the price and calculator tools, then ask a question that forces all three (get price → multiply by quantity → convert). Fill in the four TODOs and run it.

In [ ]:
# Tools the agent already knows (from section 3.3):
def get_stock_price(ticker: str):
    prices = {"NVDA": 125.50, "GOOG": 178.20, "AAPL": 229.00}
    return {"ticker": ticker.upper(), "price_usd": prices.get(ticker.upper(), 0.0)}

def calculator(expression: str):
    return {"expression": expression, "result": eval(expression)}  # demo only

# TODO 1: finish the third tool — convert a USD amount to another currency.
def convert_currency(amount_usd: float, to: str):
    rates = {"EUR": 0.92, "GBP": 0.79, "UAH": 41.0}
    ...   # return e.g. {"amount": amount_usd * rates[to], "currency": to}

TOOLS = [
    {"type": "function", "function": {
        "name": "get_stock_price",
        "description": "Get the latest share price (USD) for a ticker symbol",
        "parameters": {"type": "object",
            "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression like '125.5 * 10'",
        "parameters": {"type": "object",
            "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    # TODO 2: add the function spec for convert_currency here
]

# TODO 3: register the new tool so the loop can dispatch to it
DISPATCH = {"get_stock_price": get_stock_price, "calculator": calculator}

# TODO 4: ask something that needs all three tools, then run it:
# print(run_agent_local("How much is 10 shares of NVDA worth in euros?", TOOLS, DISPATCH))

**▶️ Expected output** (after completing TODO 1–4 and uncommenting the last line)

```
  [step 0] -> get_stock_price({'ticker': 'NVDA'})
  [step 1] -> calculator({'expression': '125.5 * 10'})
  [step 2] -> convert_currency({'amount_usd': 1255.0, 'to': 'EUR'})
10 shares of NVDA are worth about €1,154.60 (1255 USD × 0.92).
```

The model chained all three tools on its own. Exact wording, the order of the middle steps, and rounding will vary between runs and models — what matters is that **all three tools get called** and the final number is right.

### Task 2 — Structured output, locally

Section 3.4 used OpenAI's schema-guaranteed parsing. Ollama does the same trick with a JSON schema. Define a Pydantic model for a product review and extract it from the paragraph below — the reply is guaranteed to match your schema.

In [ ]:
import ollama
from pydantic import BaseModel

# TODO 1: define the schema — sentiment (str), score (int, 1-5),
#         pros (list[str]), cons (list[str])
class Review(BaseModel):
    ...

text = ("Battery life is fantastic and it's super light, but the camera is "
        "mediocre in low light and the price is a bit high.")

# TODO 2: call ollama with format=<your schema>, then validate the JSON reply.
# resp = ollama.chat(
#     model=OLLAMA_MODEL,
#     messages=[{"role": "user", "content": f"Extract a product review as JSON:\n{text}"}],
#     format=Review.model_json_schema())
# review = Review.model_validate_json(resp.message.content)
# print(review)

**▶️ Expected output** (after completing TODO 1–2)

```
sentiment='mixed' score=3 pros=['fantastic battery life', 'lightweight'] cons=['mediocre low-light camera', 'price a bit high']
```

`review` is a real, validated `Review` object — the JSON came back matching your schema exactly. The phrasing of the pros/cons will differ per run; the structure won't.

### Task 3 — Extend the MCP server (no model needed)

Add a second tool to the weather server, then use the MCP client from section 5.2 to confirm **both** tools are discovered and call the new one. This exercises the protocol directly — no LLM involved. We write to a separate file (`weather_server_hw.py`) so the section-5 demo server stays intact.

In [9]:
%%writefile weather_server_hw.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-hw")

@mcp.tool()
def get_forecast(city: str) -> str:
    """Return a short weather forecast for a city."""
    data = {"Kyiv": "18C, partly cloudy", "London": "12C, rain", "Tokyo": "24C, clear"}
    return data.get(city, f"No forecast available for {city}")

# TODO 1: add a get_air_quality(city) tool that returns an AQI string per city.
@mcp.tool()
def get_air_quality(city: str) -> str:
    """Return the air-quality index (AQI) for a city."""
    ...   # return e.g. {"Kyiv": "AQI 42 (good)"}.get(city, "unknown")

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing weather_server_hw.py


**▶️ Expected output**

```
Writing weather_server_hw.py
```

(`Overwriting weather_server_hw.py` if you run it again.) The cell just saves the file — it does **not** start the server here.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_hw = StdioServerParameters(command="python", args=["weather_server_hw.py"])

async def check():
    async with stdio_client(server_hw) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Discovered tools:", [t.name for t in tools.tools])  # expect BOTH now
            # TODO 2: call get_air_quality for a city and print result.content[0].text
            ...

await check()   # in a .py script: asyncio.run(check())

**▶️ Expected output** (after completing TODO 2)

```
Discovered tools: ['get_forecast', 'get_air_quality']
Kyiv air quality: AQI 42 (good)
```

Both tools are now visible over the protocol, and the new one returns your data — all with no LLM involved.

### Task 4 — Reading (no code)

- Skim the MCP intro: https://modelcontextprotocol.io
- Hugging Face text-classification fine-tuning guide: https://huggingface.co/docs/transformers/tasks/sequence_classification

**Stretch (optional):** combine Task 1 and Task 3 — point `run_agent_local` at the MCP tools (adapt `run_mcp_agent` from section 5.3 to use `ollama_client` instead of `client`) so a *local* model answers "Is the air in Kyiv safe today?" by calling your MCP server.